# 42. CD-OPE-S Pretrained 학습실험

A2-A5를 seed 반복으로 학습합니다. 기본값은 안전하게 `RUN_TRAIN=False`입니다.
실제 실행 시 `RUN_TRAIN=True`, `TRAIN_SEEDS=[0,1,2]`, `EPOCHS=30`으로 둡니다.

비교군:

- A2: `g_cd`
- A3: `gl_cd`
- A4: `gl_cd_consistency`
- A5: `gl_cd_consistency_style`

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 42-1. manifest와 실험 계획

In [2]:
manifests = create_ch4_manifests(max_per_cell=None, seed=41)
run_root = paths.runs_root / "cd_ope_s" / "runs"

variants = [
    {
        "variant": "g_cd",
        "use_global": True,
        "use_local": False,
        "consistency": False,
        "style_suppression": False,
    },
    {
        "variant": "gl_cd",
        "use_global": True,
        "use_local": True,
        "consistency": False,
        "style_suppression": False,
    },
    {
        "variant": "gl_cd_consistency",
        "use_global": True,
        "use_local": True,
        "consistency": True,
        "style_suppression": False,
    },
    {
        "variant": "gl_cd_consistency_style",
        "use_global": True,
        "use_local": True,
        "consistency": True,
        "style_suppression": True,
    },
]
display(pd.DataFrame(variants))

,variant,use_global,use_local,consistency,style_suppression
0,g_cd,True,False,False,False
1,gl_cd,True,True,False,False
2,gl_cd_consistency,True,True,True,False
3,gl_cd_consistency_style,True,True,True,True


## 42-2. 학습 실행

In [3]:
RUN_TRAIN = True
TRAIN_SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH_SIZE = 8
BASE_LR = 1e-4
BRANCH_LR_MULT = 3.0

if RUN_TRAIN:
    for variant in variants:
        for seed in TRAIN_SEEDS:
            run_dir = run_root / variant["variant"] / f"seed_{seed}"
            if (run_dir / "sample_metrics.csv").exists():
                print("skip existing:", run_dir)
                continue
            print("training:", variant["variant"], "seed", seed)
            train_cd_ope_s_experiment(
                manifests["train"],
                manifests["eval_matched"],
                run_dir,
                variant=variant["variant"],
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                base_lr=BASE_LR,
                branch_lr_mult=BRANCH_LR_MULT,
                seed=seed,
                use_global=variant["use_global"],
                use_local=variant["use_local"],
                consistency=variant["consistency"],
                style_suppression=variant["style_suppression"],
                local_kernel_size=15,
                consistency_weight=0.25,
                consistency_ce_weight=0.5,
                style_weight=0.05,
                ramp_epochs=5,
            )
else:
    print("RUN_TRAIN=False: 설정을 확인했습니다. 실제 학습은 True로 바꾸고 실행하세요.")

training: g_cd seed 0


C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 13703.22it/s]


[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


...

epoch 26 | loss=0.1016 | ce=0.0309 | cons=0.0003 | style=1.1030 | ag=0.0605 | al=0.0282


epoch 27 | loss=0.0999 | ce=0.0297 | cons=0.0003 | style=1.1019 | ag=0.0624 | al=0.0276


epoch 28 | loss=0.0979 | ce=0.0282 | cons=0.0002 | style=1.1011 | ag=0.0635 | al=0.0283


epoch 29 | loss=0.0955 | ce=0.0269 | cons=0.0002 | style=1.1022 | ag=0.0644 | al=0.0284


epoch 30 | loss=0.0939 | ce=0.0256 | cons=0.0002 | style=1.1017 | ag=0.0652 | al=0.0288


## 42-3. 결과 수집

In [4]:
seed_metrics = collect_ch4_cd_ope_metrics(run_root=run_root)
if seed_metrics.empty:
    print("아직 CD-OPE-S 학습 run이 없습니다.")
else:
    display(seed_metrics)
    display(pd.read_csv(paths.runs_root / "cd_ope_s" / "cd_ope_s_summary.csv"))

,variant,model_seed,run_dir,mean_dice,seen_color_dice,heldout_color_dice,red_dice,purple_dice,worst_combo_dice,target_fnr,alpha_global,alpha_local
0,g_cd,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.555547,0.558469,0.551165,0.530599,0.571731,0.160530,0.488617,0.061157,0.000000
1,g_cd,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.593736,0.621262,0.552447,0.495976,0.608918,0.205672,0.429164,0.040685,0.000000
2,g_cd,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.511495,0.526553,0.488908,0.478978,0.498838,0.101797,0.540872,0.056230,0.000000
3,gl_cd,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.525958,0.560491,0.474159,0.411909,0.536410,0.189626,0.529936,0.052356,0.022873
4,gl_cd,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.485473,0.535433,0.410532,0.382798,0.438267,0.142344,0.562911,0.045000,-0.060026
5,gl_cd,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.501366,0.514625,0.481477,0.488751,0.474204,0.143862,0.547570,0.065900,0.026714
6,gl_cd_consistency,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.603256,0.626133,0.568940,0.503065,0.634816,0.240694,0.436429,0.051055,0.025223
7,gl_cd_consistency,1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.566173,0.595692,0.521896,0.486591,0.557201,0.181444,0.478784,0.041712,-0.065609
8,gl_cd_consistency,2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.608428,0.619006,0.592562,0.581388,0.603735,0.232915,0.417324,0.069178,0.021453
9,gl_cd_consistency_style,0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,0.557728,0.578172,0.527062,0.481942,0.572182,0.164774,0.498809,0.066665,0.021097


,variant,mean_dice_mean,mean_dice_std,mean_dice_count,seen_color_dice_mean,seen_color_dice_std,seen_color_dice_count,heldout_color_dice_mean,heldout_color_dice_std,heldout_color_dice_count,red_dice_mean,red_dice_std,red_dice_count,worst_combo_dice_mean,worst_combo_dice_std,worst_combo_dice_count,target_fnr_mean,target_fnr_std,target_fnr_count
0,g_cd,0.553593,0.041156,3,0.568761,0.048186,3,0.530840,0.036320,3,0.501851,0.026307,3,0.156000,0.052085,3,0.486218,0.055893,3
1,gl_cd,0.504266,0.020398,3,0.536850,0.022966,3,0.455390,0.039020,3,0.427819,0.054739,3,0.158611,0.026871,3,0.546806,0.016501,3
2,gl_cd_consistency,0.592619,0.023048,3,0.613610,0.015922,3,0.561133,0.035974,3,0.523682,0.050650,3,0.218351,0.032198,3,0.444179,0.031454,3
3,gl_cd_consistency_style,0.565786,0.036325,3,0.585758,0.033033,3,0.535829,0.041266,3,0.502915,0.025490,3,0.166242,0.059161,3,0.485080,0.039664,3
